<p align="right"><a href="./C2_W2_Backprop.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab: 用计算图做反向传播（Back propagation using a computation graph）
做完本实验，你会理解大多数机器学习框架用到的一个关键算法。梯度下降需要代价关于网络中每个参数的导数。神经网络可能有数百万甚至数十亿个参数。*反向传播（back propagation）* 算法就是用来计算这些导数的，而 *计算图（Computation graphs）* 用来简化这个运算。下面我们深入看看。

In [1]:
from sympy import *
import numpy as np
import re
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import TextBox
from matplotlib.widgets import Button
import ipywidgets as widgets
from lab_utils_backprop import *

## 计算图（Computation Graph）
计算图通过把复杂导数拆成更小的步骤来简化其计算。我们看看它是怎么工作的。

我们来计算这个稍复杂表达式的导数：$J = (2+3w)^2$。我们想求 $J$ 关于 $w$ 的导数，即 $\frac{\partial J}{\partial w}$。

In [ ]:
plt.close("all")
plt_network(config_nw0, "./images/C2_W2_BP_network0.PNG")

上面可以看到，我们把表达式拆成了两个可以独立处理的节点。如果你已经从课程里很好地理解了这个过程，可以直接去填上图中的方框：先从左到右填蓝色框，再从右到左填绿色框。
如果值填对了，会显示为绿色或蓝色；填错了会显示为红色。注意，这个交互式图形不太稳健，如果界面出问题，重新运行上面的 cell 来重启。

如果你对这个过程没把握，下面我们会一步步做这个例子。

### 前向传播（Forward Propagation）
我们来沿前向方向计算这些值。

>关于这一节的一点说明：它使用全局变量，并随计算推进复用它们。如果你不按顺序运行 cell，可能得到奇怪的结果。如果出现这种情况，回到这一点按顺序运行。

In [ ]:
w = 3
a = 2+3*w
J = a**2
print(f"a = {a}, J = {J}")

你可以把这些值填进上面的蓝色框。

### 反向传播（Backprop）
<img align="left" src="./images/C2_W2_BP_network0_j.PNG"     style=" width:100px; padding: 10px 20px; " > Backprop 是我们用来计算导数的算法。如课程所述，backprop 从右开始、向左移动。第一个要考虑的节点是 $J = a^2 $，第一步是求 $\frac{\partial J}{\partial a}$

### $\frac{\partial J}{\partial a}$
#### 算术方式
通过"$a$ 有一个小变化时 $J$ 如何变化"来求 $\frac{\partial J}{\partial a}$。这在导数那个选修实验里有详细描述。

In [ ]:
a_epsilon = a + 0.001       # a epsilon
J_epsilon = a_epsilon**2    # J_epsilon
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k} ")

$\frac{\partial J}{\partial a}$ 是 22，也就是 $2\times a$。我们的结果不恰好是 $2 \times a$，因为我们的 epsilon 值不是无穷小。
#### 符号方式
现在，我们像在导数选修实验里那样用 SymPy 符号化地计算导数。我们会在变量名前加一个 's'，表示这是一个 *符号（symbolic）* 变量。

In [ ]:
sw,sJ,sa = symbols('w,J,a')
sJ = sa**2
sJ

In [ ]:
sJ.subs([(sa,a)])

In [ ]:
dJ_da = diff(sJ, sa)
dJ_da

所以，$\frac{\partial J}{\partial a} = 2a$。当 $a=11$ 时，$\frac{\partial J}{\partial a} = 22$。这与上面的算术计算一致。
如果还没填，你可以回到上面的图，填上 $\frac{\partial J}{\partial a}$ 的值。

### $\frac{\partial J}{\partial w}$
<img align="left" src="./images/C2_W2_BP_network0_a.PNG"     style=" width:100px; padding: 10px 20px; " >  从右向左，下一个要计算的值是 $\frac{\partial J}{\partial w}$。为此，我们先要算 $\frac{\partial a}{\partial w}$，它描述当输入 $w$ 有一个小变化时，该节点的输出 $a$ 如何变化。

#### 算术方式
通过"$w$ 有一个小变化时 $a$ 如何变化"来求 $\frac{\partial a}{\partial w}$。

In [ ]:
w_epsilon = w + 0.001       # a  plus a small value, epsilon
a_epsilon = 2 + 3*w_epsilon
k = (a_epsilon - a)/0.001   # difference divided by epsilon
print(f"a = {a}, a_epsilon = {a_epsilon}, da_dw ~= k = {k} ")

算术地算出 $\frac{\partial a}{\partial w} \approx 3$。我们用 SymPy 试试。

In [ ]:
sa = 2 + 3*sw
sa

In [ ]:
da_dw = diff(sa,sw)
da_dw

>下一步是有趣的部分：
> - 我们知道 $w$ 的一个小变化会使 $a$ 变化其 3 倍。
> - 我们知道 $a$ 的一个小变化会使 $J$ 变化其 $2\times a$ 倍。（本例中 a=11）
 于是把这些合起来，
> - 我们知道 $w$ 的一个小变化会使 $J$ 变化其 $3 \times 2\times a$ 倍。
>
> 这种级联的变化叫做 *链式法则（the chain rule）*，可写成：
 $$\frac{\partial J}{\partial w} = \frac{\partial a}{\partial w} \frac{\partial J}{\partial a} $$

如果不清楚，值得花点时间想通它，这是一个关键要点。

我们来算算看：

In [ ]:
dJ_dw = da_dw * dJ_da
dJ_dw

本例中 $a$ 是 11，所以 $\frac{\partial J}{\partial w} = 66$。我们可以用算术检验：

In [ ]:
w_epsilon = w + 0.001
a_epsilon = 2 + 3*w_epsilon
J_epsilon = a_epsilon**2
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k} ")

好！如果还没填，你现在可以把 $\frac{\partial a}{\partial w}$ 和 $\frac{\partial J}{\partial w}$ 的值填进图里。

**另一种视角**
可以这样可视化这些级联的变化：
<img align="center" src="./images/C2_W2_BP_network0_diff.PNG"  style=" width:500px; padding: 10px 20px; " >
$w$ 的一个小变化乘以 $\frac{\partial a}{\partial w}$，得到一个大 3 倍的变化；这个更大的变化再乘以 $\frac{\partial J}{\partial a}$，得到一个现在大 $3 \times 22 = 66$ 倍的变化。

## 一个简单神经网络的计算图
下面是课程里用的那个神经网络的图，取了不同的值。试着填上方框里的值。注意，这个交互式图形不太稳健，如果界面出问题，重新运行下面的 cell 来重启。

In [ ]:
plt.close("all")
plt_network(config_nw1, "./images/C2_W2_BP_network1.PNG")

下面，我们会详细走一遍填上面这张计算图所需的计算。从前向路径开始。

### 前向传播（Forward propagation）
前向路径的计算就是你最近为神经网络学过的那些。你可以把下面的值与你为上图算的值作比较。

In [ ]:
# Inputs and parameters
x = 2
w = -2
b = 8
y = 1
# calculate per step values   
c = w * x
a = c + b
d = a - y
J = d**2/2
print(f"J={J}, d={d}, a={a}, c={c}")

### 反向传播（Backprop）
<img align="left" src="./images/C2_W2_BP_network1_jdsq.PNG"     style=" width:100px; padding: 10px 20px; " > 如课程所述，backprop 从右开始、向左移动。第一个要考虑的节点是 $J = \frac{1}{2}d^2 $，第一步是求 $\frac{\partial J}{\partial d}$

### $\frac{\partial J}{\partial d}$

#### 算术方式
通过"$d$ 有一个小变化时 $J$ 如何变化"来求 $\frac{\partial J}{\partial d}$。

In [ ]:
d_epsilon = d + 0.001
J_epsilon = d_epsilon**2/2
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dd ~= k = {k} ")

$\frac{\partial J}{\partial d}$ 是 3，也就是 $d$ 的值。我们的结果不恰好是 $d$，因为我们的 epsilon 值不是无穷小。
#### 符号方式
现在，我们像在导数选修实验里那样用 SymPy 符号化地计算导数。我们会在变量名前加一个 's'，表示这是一个 *符号* 变量。

In [ ]:
sx,sw,sb,sy,sJ = symbols('x,w,b,y,J')
sa, sc, sd = symbols('a,c,d')
sJ = sd**2/2
sJ

In [ ]:
sJ.subs([(sd,d)])

In [ ]:
dJ_dd = diff(sJ, sd)
dJ_dd

所以，$\frac{\partial J}{\partial d}$ = d。当 $d=3$ 时，$\frac{\partial J}{\partial d}$ = 3。这与上面的算术计算一致。
如果还没填，你可以回到上面的图，填上 $\frac{\partial J}{\partial d}$ 的值。

### $\frac{\partial J}{\partial a}$
<img align="left" src="./images/C2_W2_BP_network1_d.PNG"     style=" width:100px; padding: 10px 20px; " >  从右向左，下一个要计算的值是 $\frac{\partial J}{\partial a}$。为此，我们先要算 $\frac{\partial d}{\partial a}$，它描述当输入 $a$ 有一个小变化时该节点的输出如何变化。（注意我们不关心 $y$ 变化时输出如何变化，因为 $y$ 不是参数。）

#### 算术方式
通过"$a$ 有一个小变化时 $d$ 如何变化"来求 $\frac{\partial d}{\partial a}$。

In [ ]:
a_epsilon = a + 0.001         # a  plus a small value
d_epsilon = a_epsilon - y
k = (d_epsilon - d)/0.001   # difference divided by epsilon
print(f"d = {d}, d_epsilon = {d_epsilon}, dd_da ~= k = {k} ")

算术地算出 $\frac{\partial d}{\partial a} \approx 1$。我们用 SymPy 试试。
#### 符号方式

In [ ]:
sd = sa - sy
sd

In [ ]:
dd_da = diff(sd,sa)
dd_da

算术地算出 $\frac{\partial d}{\partial a}$ 也等于 1。
>下一步又是那个有趣的部分，在本例中再次出现：
> - 我们知道 $a$ 的一个小变化会使 $d$ 变化其 1 倍。
> - 我们知道 $d$ 的一个小变化会使 $J$ 变化其 $d$ 倍。（本例中 d=3）
 于是把这些合起来，
> - 我们知道 $a$ 的一个小变化会使 $J$ 变化其 $1\times d$ 倍。
>
>这又是 *链式法则*，可写成：
 $$\frac{\partial J}{\partial a} = \frac{\partial d}{\partial a} \frac{\partial J}{\partial d} $$

我们来算算看：

In [ ]:
dJ_da = dd_da * dJ_dd
dJ_da

本例中 $d$ 是 3，所以 $\frac{\partial J}{\partial a} = 3$。我们可以用算术检验：

In [ ]:
a_epsilon = a + 0.001
d_epsilon = a_epsilon - y
J_epsilon = d_epsilon**2/2
k = (J_epsilon - J)/0.001   
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k} ")

好，它们一致！如果还没填，你现在可以把 $\frac{\partial d}{\partial a}$ 和 $\frac{\partial J}{\partial a}$ 的值填进图里。

> **backprop 的步骤**
>既然你已经走过了几个节点，我们可以写下基本方法：\
> 从右向左，对每个节点：
>- 计算该节点的局部导数（local derivative）
>- 用链式法则，把它与"代价关于右边那个节点的导数"结合。

'局部导数' 指的是当前节点的输出关于其所有输入或参数的导数。

我们继续做下去。既然你已熟悉这个方法，接下来会写得简略一些。

### $\frac{\partial J}{\partial c}$,  $\frac{\partial J}{\partial b}$
<img align="left" src="./images/C2_W2_BP_network1_a.PNG"     style=" width:100px; padding: 10px 20px; " >下一个节点有两个我们关心的导数。我们需要算 $\frac{\partial J}{\partial c}$ 以便向左传播，还想算 $\frac{\partial J}{\partial b}$。求代价关于参数 $w$ 和 $b$ 的导数正是 backprop 的目标。我们先求局部导数 $\frac{\partial a}{\partial c}$ 和 $\frac{\partial a}{\partial b}$，再把它们与从右边来的导数 $\frac{\partial J}{\partial a}$ 结合。

In [ ]:
# calculate the local derivatives da_dc, da_db
sa = sc + sb
sa

In [ ]:
da_dc = diff(sa,sc)
da_db = diff(sa,sb)
print(da_dc, da_db)

In [ ]:
dJ_dc = da_dc * dJ_da
dJ_db = da_db * dJ_da
print(f"dJ_dc = {dJ_dc},  dJ_db = {dJ_db}")

在我们的例子里，d = 3

###  $\frac{\partial J}{\partial w}$
<img align="left" src="./images/C2_W2_BP_network1_c.PNG"     style=" width:100px; padding: 10px 20px; " > 本例的最后一个节点计算 `c`。这里我们关心 J 关于参数 w 如何变化。我们不会反向传播到输入 $x$，所以不关心 $\frac{\partial J}{\partial x}$。我们先算 $\frac{\partial c}{\partial w}$。

In [ ]:
# calculate the local derivative
sc = sw * sx
sc

In [ ]:
dc_dw = diff(sc,sw)
dc_dw

这个导数比上一个稍有意思，它会随 $x$ 的值而变。本例中它是 2。

把它与 $\frac{\partial J}{\partial c}$ 结合，求出 $\frac{\partial J}{\partial w}$。

In [ ]:
dJ_dw = dc_dw * dJ_dc
dJ_dw

In [ ]:
print(f"dJ_dw = {dJ_dw.subs([(sd,d),(sx,x)])}")

本例中 $d=3$，所以 $\frac{\partial J}{\partial w} = 6$。
我们用算术检验：

In [ ]:
J_epsilon = ((w+0.001)*x+b - y)**2/2
k = (J_epsilon - J)/0.001  
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k} ")

它们一致！很好。如果还没填，你可以把 $\frac{\partial J}{\partial w}$ 加进上面的图里，我们的分析就完成了。

## 恭喜！
你已经走完了一个用计算图做反向传播的例子。按照同样的逐节点方法，你可以把它应用到更大的例子上。